# BQML と AutoML を使用した時系列予測

**学習目標**
 1. `CREATE MODEL` を使用して BQML で分類時系列モデルを作成する方法を学ぶ。
 2. BQML を使用して線形回帰時系列モデルを作成する方法を学ぶ。
 3. AutoML Tables を使用して BigQuery 内のデータから時系列モデルを構築する方法を学ぶ。

## 環境変数の設定と必要なライブラリの読み込み

In [ ]:
PROJECT = !(gcloud config get-value core/project)
PROJECT = PROJECT[0]
%env PROJECT = {PROJECT}
%env REGION = "us-central1"

%load_ext google.cloud.bigquery

## データセットの作成

In [ ]:
from google.cloud import bigquery
from IPython import get_ipython

bq = bigquery.Client(project=PROJECT)


def create_dataset():
    dataset = bigquery.Dataset(bq.dataset("stock_market"))
    try:
        bq.create_dataset(dataset)  # データセットがすでに存在する場合は失敗します。
        print("Dataset created")
    except:
        print("Dataset already exists")


def create_features_table():
    error = None
    try:
        bq.query(
            """
        CREATE TABLE stock_market.eps_percent_change_sp500
        AS
        SELECT *
        FROM `stock_market.eps_percent_change_sp500`
        """
        ).to_dataframe()
    except Exception as e:
        error = str(e)
    if error is None:
        print("Table created")
    elif "Already Exists" in error:
        print("Table already exists.")
    else:
        print(error)
        raise Exception("Table was not created.")


create_dataset()
create_features_table()

## データセットの確認

前のラボでデータを作成しました。前のノートブックを実行していない場合は、[2_feature_engineering.ipynb](../solutions/2_feature_engineering.ipynb) に戻ってデータを作成してください。このデータをモデリングに使用し、BigQuery 内のテーブルとして保存します。

すべてが期待どおりであることを確認するために、そのテーブルをもう一度調べてみましょう。その後、このテーブルを使用して BigQuery ML でモデルを構築します。

In [ ]:
%%bigquery --project $PROJECT
#standardSQL
SELECT
  *
FROM
  stock_market.eps_percent_change_sp500
LIMIT
  10

## BQML の使用

### `direction` に対する分類モデルの作成

モデルを作成する手順:
1. `CREATE MODEL` を使用し、生成されるモデルの出力先テーブルを指定します。または、既存のモデルの上書きを許可する `CREATE OR REPLACE MODEL` を使用することもできます。
2. `OPTIONS` を使用してモデルタイプ（linear_reg または logistic_reg）を指定します。正規化や学習率など、[指定可能なオプション](https://cloud.google.com/bigquery/docs/reference/standard-sql/bigqueryml-syntax-create#model_option_list) は他にも多数ありますが、ここではデフォルト値を使用します。
3. トレーニング データを取得するクエリを指定します。

別の例については、[このチュートリアルのステップ 2](https://cloud.google.com/bigquery/docs/bigqueryml-natality) を参照してください。

**クエリの完了には約 2 分かかります**


まず、各銘柄の株価の動向（`direction`）を予測する分類モデルの作成から始めます。

`symbol` の値を使用してランダム分割を行います。約 500 種類の異なる値がある中で、`ABS(MOD(FARM_FINGERPRINT(symbol), 15)) = 1` を使用すると 30 個の異なる `symbol` の値が得られ、これは約 171,000 件のトレーニング サンプルに相当します。そのうち 70% をトレーニング用とすると、約 110,000 件のトレーニング サンプルに基づいてモデルを構築することになります。

#### ラボタスク #1a: BQML を使用したモデルの作成
BQML の `CREATE OR REPLACE MODEL` を使用して、`percent_change_sp500` テーブルの特徴量を使用して銘柄の `direction` を予測する分類モデルをトレーニングします。正しい構文については、[BQML モデルの作成に関するドキュメント](https://cloud.google.com/bigquery-ml/docs/bigqueryml-natality#step_two_create_your_model) を参照してください。サブサンプルでトレーニングするために `ABS(MOD(FARM_FINGERPRINT(symbol), 15)) = 1` を使用してください。

In [ ]:
%%bigquery --project $PROJECT
#standardSQL
CREATE OR REPLACE MODEL
  # TODO: ここにコードを記述してください
  -- トレーニング データを取得するクエリ
SELECT
  # TODO: ここにコードを記述してください
FROM
  `stock_market.eps_percent_change_sp500`
WHERE
  # TODO: Your code goes here

## トレーニング統計の取得とトレーニング情報の確認

モデルを作成した後、[`ML.EVALUATE` 関数](https://cloud.google.com/bigquery-ml/docs/bigqueryml-natality#step_four_evaluate_your_model) を使用してパフォーマンスを評価できます。このコマンドを使用すると、分類モデルの適合率（precision）、再現率（recall）、正解率（accuracy）、F1 スコア、および AUC を確認できます。

#### ラボタスク #1b: BQML モデルの評価
BQML の `EVALUATE` を使用して、検証セットに対するモデルのパフォーマンスを評価します。クエリは [この例](https://cloud.google.com/bigquery-ml/docs/bigqueryml-natality#step_four_evaluate_your_model) に類似したものになります。

In [ ]:
%%bigquery --project $PROJECT
#standardSQL
SELECT
  *
FROM
  # TODO: ここにコードを記述してください。
  

BigQuery によって収集されたトレーニング統計を確認することもできます。トレーニング結果を表示するには、[`ML.TRAINING_INFO`](https://cloud.google.com/bigquery/docs/reference/standard-sql/bigqueryml-syntax-train) 関数を使用します。

#### ラボタスク #1c: BQML のトレーニング情報の確認
BQML の `TRAINING_INFO` を使用して、上記で実行したトレーニング ジョブの統計情報を確認します。

In [ ]:
%%bigquery --project $PROJECT
#standardSQL
SELECT
  *
FROM
  # TODO: Your code goes here

### 単純なベンチマークとの比較

モデルのパフォーマンスを評価するもう 1 つの方法は、単純なベンチマークと比較することです。これは、単に多数派のクラスを予測するという単純な（ナイーブな）戦略を使用した場合にどのような正解率が得られるかを確認することで行えます。トレーニング データセットでは、多数派クラスは「STAY」です。以下のクエリで、この単純な戦略が評価セットでどのように機能するかを確認できます。

In [ ]:
%%bigquery --project $PROJECT
#standardSQL
WITH
  eval_data AS (
  SELECT
    symbol,
    Date,
    Open,
    close_MIN_prior_5_days,
    close_MIN_prior_20_days,
    close_MIN_prior_260_days,
    close_MAX_prior_5_days,
    close_MAX_prior_20_days,
    close_MAX_prior_260_days,
    close_AVG_prior_5_days,
    close_AVG_prior_20_days,
    close_AVG_prior_260_days,
    close_STDDEV_prior_5_days,
    close_STDDEV_prior_20_days,
    close_STDDEV_prior_260_days,
    direction
  FROM
    `stock_market.eps_percent_change_sp500`
  WHERE
    tomorrow_close IS NOT NULL
    AND ABS(MOD(FARM_FINGERPRINT(symbol), 15)) = 1
    AND ABS(MOD(FARM_FINGERPRINT(symbol), 15 * 100)) > 15 * 70
    AND ABS(MOD(FARM_FINGERPRINT(symbol), 15 * 100)) <= 15 * 85)
SELECT
  direction,
  (COUNT(direction)* 100 / (
    SELECT
      COUNT(*)
    FROM
      eval_data)) AS percentage
FROM
  eval_data
GROUP BY
  direction

したがって、多数派クラスを単純に推測する戦略では、評価データセットでの正解率は 0.5509 となり、BQML モデルをわずかに下回ります。

### `normalized change` に対する回帰モデルの作成

BigQuery を使用して、各銘柄の正規化された変化量（normalized change）を予測する回帰モデルをトレーニングすることもできます。BigQuery でこれを行うには、`CREATE OR REPLACE MODEL` を呼び出す際の OPTIONS を変更するだけです。これにより、株価が上昇するか、下落するか、横ばいかを予測するだけでなく、より詳細な予測が得られます。このように、ビジネスのニーズに応じて、この問題を回帰問題としても分類問題としても扱うことができます。


#### ラボタスク #2a: BQML での回帰モデルの作成
BQML の `CREATE OR REPLACE MODEL` を使用して別のモデル（今回は回帰モデル）をトレーニングし、上記で使用したのと同じ特徴量に基づいて特定の銘柄の `normalized_change` を予測します。

In [ ]:
%%bigquery --project $PROJECT
#standardSQL
CREATE OR REPLACE MODEL
  # TODO: ここにコードを記述してください
  -- トレーニング データを取得するクエリ
SELECT
  # TODO: ここにコードを記述してください
FROM
  `stock_market.eps_percent_change_sp500`
WHERE
  # TODO: Your code goes here

前述と同様に、回帰モデルの評価メトリクスを確認し、BigQuery でトレーニング統計を調べることができます。

In [ ]:
%%bigquery --project $PROJECT
#standardSQL
SELECT
  *
FROM
  ML.EVALUATE(MODEL `stock_market.price_model`,
    (
    SELECT
      symbol,
      Date,
      Open,
      close_MIN_prior_5_days,
      close_MIN_prior_20_days,
      close_MIN_prior_260_days,
      close_MAX_prior_5_days,
      close_MAX_prior_20_days,
      close_MAX_prior_260_days,
      close_AVG_prior_5_days,
      close_AVG_prior_20_days,
      close_AVG_prior_260_days,
      close_STDDEV_prior_5_days,
      close_STDDEV_prior_20_days,
      close_STDDEV_prior_260_days,
      normalized_change
    FROM
      `stock_market.eps_percent_change_sp500`
    WHERE
      normalized_change IS NOT NULL
      AND ABS(MOD(FARM_FINGERPRINT(symbol), 15)) = 1
      AND ABS(MOD(FARM_FINGERPRINT(symbol), 15 * 100)) > 15 * 70
      AND ABS(MOD(FARM_FINGERPRINT(symbol), 15 * 100)) <= 15 * 85))

In [ ]:
%%bigquery --project $PROJECT
#standardSQL
SELECT
  *
FROM
  ML.TRAINING_INFO(MODEL `stock_market.price_model`)
ORDER BY iteration

#### ラボタスク #3: AutoML を使用したモデルの作成
以下の手順に従って、AutoML を使用して時系列モデルを作成します。ここでは、上記と同様に `direction` を予測する分類モデルを構築する手順を説明します。同様に回帰モデルも簡単に作成できます。

## AutoML を使用した時系列モデルのトレーニング



### ステップ 1. データセットの作成

Google Cloud のナビゲーション メニューで、**Agent Platform -> データセット（Datasets）** に移動します。

**[作成]（Create）** を選択し、`stock_market` などの名前を付けます。**[データのタイプと目標を選択]（Select a data type and objective）** セクションで **[表形式]（Tabular）** を選択し、Tabular の下の **[回帰/分類]（Regression/Classification）** を選択します。下部にある **[作成]（Create）** をクリックします。

<img src='../assets/create_dataset.png' width='70%'>

### ステップ 2. データのインポート

データセットを作成したら、BigQuery を使用してデータを追加できます。**[データソースを選択]（Select a data source）** で **[BigQuery からテーブルまたはビューを選択]（Select a table or view from BigQuery）** を選択します。BigQuery パス フィールドで `eps_percent_change_sp500` を参照して指定します。

<img src='../assets/import_data.png' width='50%'>

### ステップ 3. モデルのトレーニング

データがデータセットにインポートされたら、**[統計を生成]（GENERATE STATISTICS）** をクリックして各特徴量列の基本統計量を取得できます（注: これには数分かかります）。データがインポートされたので、分類モデルのトレーニングに進むことができます。左側の列から **[新しいモデルのトレーニング]（Train a new model）** を選択します。目標を「分類（Classification）」に設定します。モデルのトレーニング方法が AutoML に設定されていることを確認します。**[続行]（Continue）** をクリックします。

<img src='../assets/set_training_objective.png' width='80%'>

[モデルの詳細]（Model details）で、ターゲット列として `direction` 列を選択します。これがモデルのトレーニング対象となるラベル列です。**[続行]（Continue）** をクリックします。

<img src='../assets/select_target_label.png' width='80%'>

[トレーニング オプション]（Training options）で、次の列**以外**のすべての列を除外します。
```
symbol
Date
Open
close_MIN_prior_5_days
close_MIN_prior_20_days
close_MIN_prior_260_days
close_MAX_prior_5_days
close_MAX_prior_20_days
close_MAX_prior_260_days
close_AVG_prior_5_days
close_AVG_prior_20_days
close_AVG_prior_260_days
close_STDDEV_prior_5_days
close_STDDEV_prior_20_days
close_STDDEV_prior_260_days
```
これらは、BQML モデルで使用したのと同じ列です。これにより、BQML モデルと AutoML モデルのパフォーマンスを比較しやすくなります。
<img src='../assets/select_training_columns.png' width='80%'>

[コンピューティングと料金]（Compute and pricing）で、**[予算]（Budget）** を 1 に設定します。わずか 1 ノード時間のトレーニングでもモデルに十分なパフォーマンスが得られます。**[早期停止を有効にする]（Enable early stopping）** を選択しておくと、改善が見られない場合にモデルのトレーニングが停止します。これで、**[トレーニングを開始]（START TRAINING）** を選択できます。モデルのトレーニングが開始されます。

<img src='../assets/select_node_hours.png' width='80%'>

### ステップ 4. モデルの評価

トレーニングには数時間かかる場合があります。しかし、トレーニングが完了すると、モデルの評価メトリクスを調べることができます。これは分類タスクであるため、しきい値を調整して、さまざまなしきい値が評価メトリクスにどのように影響するかを調査することもできます。また、そのページではモデルで使用されているさまざまな特徴量について特徴量の重要度（feature importance）を調べたり、モデル予測の混同行列（confusion matrix）を表示したりできます。

<img src='../assets/analyze_eval_metrics.png' width='80%'>

### ステップ 5. トレーニング済みモデルによる予測

モデルのトレーニングが完了したら、**[デプロイとテスト]（Deploy and test）** タブに移動し、モデルをエンドポイントにデプロイして予測をテストできるようにします。

<img src='../assets/deploy_model_endpoint.png' width='80%'>

予測を実行する際は、BigQuery テーブルまたは CSV ファイルを指定してバッチ予測ジョブを呼び出すことができます。また、単一インスタンスに対してオンライン予測を実行することもできます。

Copyright 2019 Google Inc. Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License